In [24]:
import pandas as pd
from glob import glob


import librosa, librosa.display
from pydub import AudioSegment
import numpy as np
import IPython.display as ipd
import os
import matplotlib.pyplot as plt
import whisper
import torch
import plotly.express as px
from sklearn.decomposition import PCA
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

In [ ]:
audio_files_m4a = glob('/Users/d0304693/Documents/fun/als-voice-cloning-evaluation/data/m4a/*.m4a')
print(len(audio_files_m4a))

In [ ]:
y, sr = librosa.load(audio_files_m4a[5])
pd.Series(y).plot(title='Audio Signal')

In [ ]:
print(f'Shape of audio signal: {y.shape}')

In [ ]:
data_frame_m4a = []

In [ ]:
for f in audio_files_m4a:
    y, sr = librosa.load(f)
    data_frame_m4a.append({
        'file': f,
        'duration_s': librosa.get_duration(y=y, sr=sr),
        'sample_rate': sr,
        'num_samples': len(y),
        'file_format': f.split('.')[-1],
    })
df = pd.DataFrame(data_frame_m4a)

In [ ]:
df

In [ ]:
df.describe()

### Convert to .wav format

In [ ]:
for f in audio_files_m4a:
    base_file_name = f.split('/')[-1].split('.')[0]
    audio = AudioSegment.from_file(f, format='m4a')
    audio.export(f'data/wav/{base_file_name}.wav', format='wav')

### Signal Analysis


In [ ]:
audio_files_wav = glob('/Users/d0304693/Documents/fun/als-voice-cloning-evaluation/data/wav/*.wav')
print(len(audio_files_wav))

In [ ]:
data_frame_wav = []
for f in audio_files_wav:
    y, sr = librosa.load(f)
    data_frame_wav.append({
        'file': f,
        'duration_s': librosa.get_duration(y=y, sr=sr),
        'sample_rate': sr,
        'num_samples': len(y),
        'file_format': f.split('.')[-1],
    })
df_wav = pd.DataFrame(data_frame_wav)

In [ ]:
peaks = []
for f in audio_files_wav:
    y, sr = librosa.load(f)
    peak = np.max(np.abs(y))
    peaks.append(peak)
df_wav['peak_amplitude'] = peaks

In [ ]:
df_wav['peak_amplitude'].hist(bins = 42)

In [ ]:
df_filtered = df_wav[df_wav['peak_amplitude'] > 0.9]
df_filtered

Note: Some of the audio files have very high peak amplitudes. After listening to the files, it seems that these are mostly background noise or silent recordings. Remember for the preprocessing to filter out the background noise.

In [ ]:
audio_file = glob('/Users/d0304693/Documents/fun/als-voice-cloning-evaluation/data/wav/AUDIO-2021-07-30-06-44-26.wav')
ipd.Audio(audio_file[0])

In [ ]:
y, sr = librosa.load(audio_file[0])
librosa.display.waveshow(y, sr=sr)

### Spectrogram analysis

The goal of the mel spectrogram analysis is to visualize the noises and quality degradation. This prevents the model from learning the background noise and instead focuses on cloning the patient’s natural and clear voice identity.



In [ ]:
audio_file = glob('/Users/d0304693/Documents/fun/als-voice-cloning-evaluation/data/wav/AUDIO-2021-08-16-08-21-22.wav')
ipd.Audio(audio_file[0])

In [ ]:
n_fft = 2048
hop_length = 512

for files in audio_files_wav:
    signal, sr = librosa.load(files)

    base_file_name = os.path.splitext(os.path.basename(files))[0]
    mel_signal = librosa.feature.melspectrogram(y=signal, sr=sr, hop_length=hop_length,
                                                n_fft=n_fft)
    spectrogram = np.abs(mel_signal)

    power_to_db = librosa.power_to_db(spectrogram, ref=np.max)
    plt.figure(figsize=(10,4))
    librosa.display.specshow(power_to_db, sr=sr, x_axis='time', y_axis='mel', cmap='magma',
                             hop_length=hop_length)
    plt.colorbar(format='%+2.0f dB')
    plt.title(f'Mel-Spectrogram (dB) of {base_file_name}')
    plt.xlabel('Time', fontdict=dict(size=15))
    plt.ylabel('Frequency', fontdict=dict(size=15))
    plt.tight_layout()
    plt.savefig(f'/Users/d0304693/Documents/fun/als-voice-cloning-evaluation/images/{base_file_name}')
    plt.close()

In [ ]:
spectogram_files = glob('/Users/d0304693/Documents/fun/als-voice-cloning-evaluation/images/*.png')
print(len(spectogram_files))

### Speaker Embedding

The goal is to generate a speaker embedding and project it into a 2D space for visualization. This will help to understand the speaker's voice characteristics. The expectation is that all embeddings will be similar to each other.

In [15]:
model = whisper.load_model("large-v2")

100%|█████████████████████████████████████| 2.87G/2.87G [03:31<00:00, 14.6MiB/s]


In [16]:
audio_files = glob('/home/blackshark/IdeaProjects/als-voice-cloning-evaluation/data/wav/*.wav')
audio_files.sort()

In [17]:
audio_embeddings = []
file_labels = []

In [18]:
for f in audio_files:

    audio = whisper.load_audio(f)
    audio = whisper.pad_or_trim(audio)

    mel = whisper.log_mel_spectrogram(audio).to(model.device)

    with torch.no_grad():
        embeddings = model.encoder(mel.unsqueeze(0))

    single_embedding = embeddings.mean(dim=1).squeeze().cpu().numpy()
    audio_embeddings.append(single_embedding)

In [21]:
embeddings_array = np.array(audio_embeddings)

pca = PCA(n_components=2)
reduced_embeddings = pca.fit_transform(embeddings_array)

df_pca_embeddings = pd.DataFrame(reduced_embeddings, columns=['PC1', 'PC2'])
df_pca_embeddings['file'] = [os.path.basename(f) for f in audio_files]

fig = px.scatter(df_pca_embeddings, x='PC1', y='PC2', text='file',
                 title='PCA of audio Embeddings')
fig.show()

In [23]:
embeddings_array = np.array(audio_embeddings)

pca = PCA(n_components=2)
reduced_embeddings = pca.fit_transform(embeddings_array)

df_pca_embeddings = pd.DataFrame(reduced_embeddings, columns=['PC1', 'PC2'])

fig = px.scatter(df_pca_embeddings, x='PC1', y='PC2',
                 title='PCA of audio Embeddings')
fig.show()

### Phoneme Analysis

The goal of the phoneme analysis is to understand the phonetic characteristics of the audio files. I want to ensure that all relevant sounds and phonemes are present in the audio files. This will help if the model struggels with specific words.


In [3]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model_id = "primeline/whisper-large-v3-german"
model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)
processor = AutoProcessor.from_pretrained(model_id)
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    max_new_tokens=128,
    chunk_length_s=30,
    batch_size=16,
    return_timestamps=True,
    torch_dtype=torch_dtype,
    device=device,
)

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 1259/1259 [00:01<00:00, 817.16it/s]
`torch_dtype` is deprecated! Use `dtype` instead!
Passing `generation_config` together with generation-related arguments=({'return_timestamps', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [7]:
audio = 'als-voice-cloning-evaluation/data/wav/AUDIO-2021-08-18-22-50-03.wav'
result = pipe(audio)
print(result["text"])

FileNotFoundError: [Errno 2] No such file or directory: 'als-voice-cloning-evaluation/data/wav/AUDIO-2021-08-18-22-50-03.wav'